In [1]:
import os

In [2]:
%pwd

'/home/user/Documents/Kidney_Disease Using Deep Learning/research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'/home/user/Documents/Kidney_Disease Using Deep Learning'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_batch_size: int
    params_epochs: int
    params_is_argumentation: bool
    training_data: Path
    params_learning_rate: float

In [6]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml,create_directories
import tensorflow as tf

I0000 00:00:1776964305.403532   23521 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776964305.404637   23521 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776964305.508066   23521 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776964307.850413   23521 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [ ]:
class ConfigurationManager:
    def __init__(
            self,
            config_file_path=CONFIG_FILE_PATH,
            params_file_path=PARAMS_FILE_PATH
    ):
        
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def training_config(self) -> TrainingConfig:
        training = self.config.training 
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_indigestion.unzip_dir, "kidney-ct-scan-image")
        create_directories = ([
             Path(training.root_dir)
        ])

        training_config = TrainingConfig(
          root_dir=Path(self.config.artifacts.training.root_dir),
        trained_model_path=Path(self.config.artifacts.training.trained_model_path),
    updated_base_model_path=Path(self.config.artifacts.prepare_base_model.updated_base_model_path),
    params_image_size=self.params.prepare_base_model.params_image_size,
    training_data=Path(self.config.artifacts.data_ingestion.unzip_data_file),
    params_batch_size=self.params.training.batch_size,
    params_epochs=self.params.training.epochs,
    params_is_argumentation=self.params.training.is_augmentation,
    params_learning_rate=self.params.training.learning_rate   # ✅ ADD THIS
)

        return training_config

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import time
import tensorflow as tf

In [9]:
class Training: 
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
              self.config.updated_base_model_path
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenertor(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subsets="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_argumentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenertor(
              rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

            self.train_generator = train_datagenerator.flow_from_directory(
                  directory=self.config.training_data,
                  substets="training",
                  shuffle=True,
                    **dataflow_kwargs
            )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)



    
    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )
        


In [10]:
try:
    config = ConfigurationManager()
    training_config = config.training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

2026-04-23 22:41:50,514 - INFO - cnnClassifier
2026-04-23 22:41:50,520 - INFO - cnnClassifier
2026-04-23 22:41:50,523 - INFO - cnnClassifier


BoxKeyError: "'ConfigBox' object has no attribute 'data_indigestion'"